In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar -xzf spark-3.5.8-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.8-bin-hadoop3"

In [ ]:
!ls

sample_data  spark-3.5.8-bin-hadoop3  spark-3.5.8-bin-hadoop3.tgz


In [ ]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
#spark = SparkSession.builder.master("local[*]").getOrCreate()
spark = SparkSession.builder \
                    .master("local[*]") \
                    .appName("Ejemplo") \
                    .getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = spark.read.csv(
    '/content/drive/MyDrive/airbnb-listings.csv',
    header=True,
    sep=';',
    multiLine=True,          # para textos largos
    quote='"',               # maneja comillas
    escape='"',              # evita errores de parsing
    inferSchema=True
)
df.show(10)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+--------+--------------------+--------------+------------+--------------------+--------------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------------+----------+--------------------+--------------------+------------------+------------------+--------------------+--------------------+--------------------+------------------+-------------------+-------------------------+--------------------+--------------------+-------------+----------------------+----------------------------+---------+-------------+-------+---------+--------------------+------------+-----------+------------------+------------------

El dataset presentó problemas en la estructura debido a la presencia de saltos de línea dentro de campos de texto, lo que requirió el uso del parámetro multiLine=True en PySpark para una correcta lectura.

In [ ]:
#Información general
print("Número de registros:", df.count())
print("Número de columnas:", len(df.columns))
df.printSchema()

Número de registros: 494954
Número de columnas: 89
root
 |-- ID: string (nullable = true)
 |-- Listing Url: string (nullable = true)
 |-- Scrape ID: string (nullable = true)
 |-- Last Scraped: date (nullable = true)
 |-- Name: string (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Space: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Experiences Offered: string (nullable = true)
 |-- Neighborhood Overview: string (nullable = true)
 |-- Notes: string (nullable = true)
 |-- Transit: string (nullable = true)
 |-- Access: string (nullable = true)
 |-- Interaction: string (nullable = true)
 |-- House Rules: string (nullable = true)
 |-- Thumbnail Url: string (nullable = true)
 |-- Medium Url: string (nullable = true)
 |-- Picture Url: string (nullable = true)
 |-- XL Picture Url: string (nullable = true)
 |-- Host ID: integer (nullable = true)
 |-- Host URL: string (nullable = true)
 |-- Host Name: string (nullable = true)
 |-- Host Since: date (nulla

In [ ]:
df.show(10, vertical=True, truncate=False)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Se detectó que variables numéricas relevantes como precio, número de habitaciones y baños se encuentran almacenadas como texto, lo que requiere procesos adicionales de limpieza y transformación.

# Proyecto | Lectura, escritura, archivos de Big Data PySpark

Data tomada de https://www.kaggle.com/datasets/joebeachcapital/airbnb

In [ ]:
#Valores nulos
from pyspark.sql.functions import col, sum

nulls = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

nulls.show()

nulls.toPandas().T.sort_values(by=0, ascending=False).head(10)

+---+-----------+---------+------------+----+-------+------+-----------+-------------------+---------------------+------+-------+------+-----------+-----------+-------------+----------+-----------+--------------+-------+--------+---------+----------+-------------+----------+------------------+------------------+--------------------+------------------+----------------+------------------+-------------------+-------------------------+------------------+------+-------------+----------------------+----------------------------+----+-----+-------+------+--------------+------------+-------+--------+---------+-------------+---------+------------+---------+--------+----+--------+---------+-----------+-----+------------+-------------+----------------+------------+---------------+------------+--------------+--------------+----------------+----------------+---------------+---------------+---------------+----------------+---------------------+-----------------+------------+-----------+--------------

,0
Has Availability,485647
Square Feet,482745
License,480358
Host Acceptance Rate,452696
Monthly Price,398863
Weekly Price,397207
Neighbourhood Group Cleansed,392791
Jurisdiction Names,360401
Notes,297291
Security Deposit,290942


Se identificó una alta proporción de valores nulos en múltiples variables, lo que indica posibles limitaciones en la recolección de datos o baja relevancia de ciertas variables para todos los registros.

## Transformación

In [ ]:
from pyspark.sql.functions import regexp_replace, col

df_clean = df \
    .withColumn("Bathrooms", col("Bathrooms").cast("float")) \
    .withColumn("Bedrooms", col("Bedrooms").cast("float")) \
    .withColumn("Accommodates", col("Accommodates").cast("int")) \
    .withColumn("Price", regexp_replace("Price", "[$,]", "").cast("float")) \
    .withColumn("Weekly Price", regexp_replace("Weekly Price", "[$,]", "").cast("float")) \
    .withColumn("Monthly Price", regexp_replace("Monthly Price", "[$,]", "").cast("float")) \
    .withColumn("Security Deposit", regexp_replace("Security Deposit", "[$,]", "").cast("float")) \
    .withColumn("Cleaning Fee", regexp_replace("Cleaning Fee", "[$,]", "").cast("float")) \
    .withColumn("Extra People", regexp_replace("Extra People", "[$,]", "").cast("float")) \
    .withColumn("Host Acceptance Rate", regexp_replace("Host Acceptance Rate", "%", "").cast("float")) \
    .withColumn("Host Response Rate", regexp_replace("Host Response Rate", "%", "").cast("float"))

In [ ]:
cols = [
    "Price",
    "Bedrooms",
    "Bathrooms",
    "Beds",
    "Accommodates",
    "Square Feet",
    "Number of Reviews",
    "Review Scores Rating",
    "Review Scores Cleanliness",
    "Review Scores Location",
    "Review Scores Value",
    "Host Acceptance Rate",
    "Host Response Rate",
    "Availability 30",
    "Availability 365",
    "Minimum Nights",
    "Maximum Nights"
]

In [ ]:
from pyspark.sql.functions import col, round
from pyspark.sql.types import NumericType

summary_df = df_clean.select(cols).summary()

# Identificar columnas numéricas dentro del summary
numeric_cols = [c for c in summary_df.columns if c != "summary"]

# Aplicar redondeo
for c in numeric_cols:
    summary_df = summary_df.withColumn(c, round(col(c), 2))

summary_df.show()

+-------+--------+--------+---------+--------+------------+-----------+-----------------+--------------------+-------------------------+----------------------+-------------------+--------------------+------------------+---------------+----------------+--------------+--------------+
|summary|   Price|Bedrooms|Bathrooms|    Beds|Accommodates|Square Feet|Number of Reviews|Review Scores Rating|Review Scores Cleanliness|Review Scores Location|Review Scores Value|Host Acceptance Rate|Host Response Rate|Availability 30|Availability 365|Minimum Nights|Maximum Nights|
+-------+--------+--------+---------+--------+------------+-----------+-----------------+--------------------+-------------------------+----------------------+-------------------+--------------------+------------------+---------------+----------------+--------------+--------------+
|  count|486996.0|494328.0| 493428.0|494037.0|    494891.0|    12209.0|         494952.0|            367134.0|                 366479.0|              3

In [ ]:
import matplotlib.pyplot as plt

pdf = df_clean.select(cols).dropna().toPandas()
corr = pdf.corr()
corr

plt.figure(figsize=(10,8))
plt.imshow(corr)
plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Matriz de correlación ampliada")
plt.show()

In [ ]:
pdf.hist(figsize=(12,10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
df_clean.groupBy("Country").count().orderBy("count", ascending=False).show(20, truncate=False)

+--------------+------+
|Country       |count |
+--------------+------+
|United States |134545|
|United Kingdom|61041 |
|France        |56562 |
|Spain         |45844 |
|Australia     |40693 |
|Italy         |33146 |
|Canada        |30787 |
|Germany       |20576 |
|Denmark       |20545 |
|Netherlands   |15181 |
|Austria       |7893  |
|Belgium       |7419  |
|Ireland       |6729  |
|Hong Kong     |6423  |
|Greece        |5127  |
|Switzerland   |2381  |
|China         |51    |
|NULL          |3     |
|Mexico        |2     |
|Vatican City  |2     |
+--------------+------+
only showing top 20 rows



In [ ]:
from pyspark.sql.functions import col

total = df_clean.count()

df_country_pct = df_clean.groupBy("Country").count() \
    .withColumn("percentage", (col("count")/total)*100) \
    .orderBy("count", ascending=False)

df_country_pct.show(20, truncate=False)

+--------------+------+---------------------+
|Country       |count |percentage           |
+--------------+------+---------------------+
|United States |134545|27.183334208835568   |
|United Kingdom|61041 |12.332661217001984   |
|France        |56562 |11.42772863740873    |
|Spain         |45844 |9.262274878069478    |
|Australia     |40693 |8.22157210569063     |
|Italy         |33146 |6.696783943558391    |
|Canada        |30787 |6.220173995967302    |
|Germany       |20576 |4.157153998149323    |
|Denmark       |20545 |4.150890789851179    |
|Netherlands   |15181 |3.067153715294755    |
|Austria       |7893  |1.5946936482986298   |
|Belgium       |7419  |1.498927173030221    |
|Ireland       |6729  |1.3595202786521576   |
|Hong Kong     |6423  |1.297696351580147    |
|Greece        |5127  |1.035853836922219    |
|Switzerland   |2381  |0.4810548050929985   |
|China         |51    |0.010303987845335122 |
|NULL          |3     |6.061169320785366E-4 |
|Mexico        |2     |4.040779547